# Step 1 of 3 — Data preparation (Snowflake → local files) · Iteration 4

**What this notebook produces**

| File | What it is | Who reads it |
|---|---|---|
| `local_train_data/chunk_XXXX.parquet` | Daily rows per series, `TRAIN_START` → `FORECAST_END`, ~100 dealers per file. Holds the target, the static columns and the two **scaled** per-series spend columns. `NET_SALES` is empty after `VAL_END`. | Step 2 (cache build) |
| `shared_calendar.parquet` | One row per date, `TRAIN_START` → `FORECAST_END`: the 0/1 festive and calendar flags, the **scaled** petrol index, and the hand-built calendar features | Steps 2 and 3 |
| `category_vocab.json` | For every static column, the list of categories. Position in the list = integer code. Code 0 = unknown. | Steps 2 and 3 |
| `column_roles.json` | Column roles, dates, window lengths and the **scaling constants** | Steps 2 and 3 |

**What changed vs iteration 3**
1. **New time frames** (from `Feature_details.txt`): train 2023-04-01 → 2026-06-14, validation 2026-06-15 → 2026-09-30, forecast 2026-10-01 → 2026-12-31 (92 days, so `OUTPUT_CHUNK_LENGTH = 92`).
   `VAL_START` is now the first day whose sales are **scored**. The encoder for those windows starts 365 days earlier (`VAL_SERIES_START`).
2. **Three kinds of covariate**, because the new columns do not all behave like festival flags:
   - *binary flags* (festivals, harvest windows, `IS_TUESDAY`, `IS_WEEKEND`, `POST_GST_CUT`, …): identical for every series, so they live in the shared calendar. No scaling.
   - `PETROL_PRICE_INDEX`: national, identical for every series, so it also lives in the shared calendar. **Z-scored.**
   - `TOTAL_SPEND_PER_10K` and `TOTAL_ADSTOCK_SPEND_PER_10K`: **different for every series** (they follow model family and state), so they cannot live in the shared calendar. They travel with each series' rows. **Log-transformed, then z-scored.**
3. **Scaling is done here, once** (Section 5). The model itself has no scaler: the target is raw counts under a Negative Binomial loss, and every old covariate was already between -1 and 1.
4. **New data-quality checks:** every series has rows through `FORECAST_END`; binary columns really are 0/1; no nulls in any covariate; spend is never negative; `POST_GST_CUT` switches once; the petrol values in the forecast window are shown so a carried-forward value is visible.
5. Everything from iteration 3 is kept: code book for embeddings, all-series calendar check, checks inside Snowflake before download, chunk folder wiped on each run.

In [ ]:
import os
import sys
import json
import gc

import numpy as np
import pandas as pd

sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
import snowflake_utils            # noqa: F401  (kept for parity with earlier iterations)
import Snowflake_configuration

from snowflake.snowpark.session import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import StringType

## Section 0 — Configuration
Every date and path lives here. Nothing below hard-codes a date.

**How the periods are used** (dates from `Feature_details.txt`)
- `TRAIN` (2023-04-01 → 2026-06-14): every training window's *output* (the days the loss is computed on) lies inside this range.
- `VAL` (2026-06-15 → 2026-09-30): the days whose sales are **scored** in validation. They start the day after `TRAIN_END`, so there is no overlap with training targets.
  Each validation window is 365 input days + 92 output days, so the encoder for the first scored window starts on `VAL_SERIES_START` = 2025-06-15.
  108 scored days with a 92-day window gives 17 windows per series.
- `FORECAST` (2026-10-01 → 2026-12-31): 92 days, so `OUTPUT_CHUNK_LENGTH = 92`.

In [ ]:
TABLE_NAME        = "MOP_DATABASE.SOQ.DAILY_DATA_WITH_FESTIVE_FEATURES"
VALID_SERIES_TBL  = "MOP_DATABASE.SOQ.VALID_SERIES_FOR_DAILY_MODELLING"

local_train_dir = "./local_train_data"
calendar_path   = "./shared_calendar.parquet"
vocab_path      = "./category_vocab.json"
roles_path      = "./column_roles.json"

time_col   = "CAL_DATE"
group_col  = "PARENT_DEALER_CODE_MODEL_FAMILY"
dealer_col = "PARENT_DEALER_CODE"
target_col = "NET_SALES"
FREQ       = "D"
CHUNK_SIZE = 100                      # dealers per parquet file

# ---- time frames (Feature_details.txt) ------------------------------------------
TRAIN_START    = "2023-04-01"
TRAIN_END      = "2026-06-14"
VAL_START      = "2026-06-15"         # first day whose sales are SCORED in validation
VAL_END        = "2026-09-30"
FORECAST_START = "2026-10-01"
FORECAST_END   = "2026-12-31"

INPUT_CHUNK_LENGTH  = 365
OUTPUT_CHUNK_LENGTH = (pd.Timestamp(FORECAST_END) - pd.Timestamp(FORECAST_START)).days + 1   # 92
# the encoder of the first scored validation window starts this many days before VAL_START
VAL_SERIES_START = str((pd.Timestamp(VAL_START) - pd.Timedelta(days=INPUT_CHUNK_LENGTH)).date())

# ---- column groups ---------------------------------------------------------------
# differ by series (model family x state) -> per-series covariates; log1p + z-score
SERIES_COV_COLS  = ["TOTAL_SPEND_PER_10K", "TOTAL_ADSTOCK_SPEND_PER_10K"]
# national, same value for every series -> shared calendar; z-score
SHARED_CONT_COLS = ["PETROL_PRICE_INDEX"]
# columns added for this iteration that must arrive as 0/1 flags
NEW_BINARY_COLS  = ["IS_TUESDAY", "IS_WEEKEND", "IS_RABI_HARVEST_WINDOW",
                    "IS_KHARIF_HARVEST_WINDOW", "POST_GST_CUT"]

# columns that are strings but are NOT static covariates
NON_STATIC_STRING_COLS = ["MODEL_FAMILY_CODE", "DAY_OF_THE_WEEK", group_col]
# raw calendar columns: replaced by hand-built features in Section 7
CALENDAR_RAW_COLS = ["YEAR", "MONTH", "DAY_OF_THE_MONTH", "DAY_OF_THE_WEEK"]
# columns dropped before anything else
DROP_COLS = ["FIRST_SALE_DATE", "DATASET_TYPE"]

# --- sanity checks on the configuration itself ----------------------------------
_d = {k: pd.Timestamp(v) for k, v in dict(TRAIN_START=TRAIN_START, TRAIN_END=TRAIN_END,
                                           VAL_START=VAL_START, VAL_END=VAL_END,
                                           VAL_SERIES_START=VAL_SERIES_START,
                                           FORECAST_START=FORECAST_START, FORECAST_END=FORECAST_END).items()}
assert _d["TRAIN_START"] < _d["TRAIN_END"] < _d["VAL_START"] <= _d["VAL_END"] < _d["FORECAST_START"] <= _d["FORECAST_END"], \
    "Dates must run TRAIN_START < TRAIN_END < VAL_START <= VAL_END < FORECAST_START <= FORECAST_END."
assert _d["FORECAST_START"] == _d["VAL_END"] + pd.Timedelta(days=1), \
    "History must end the day before the forecast starts."
assert _d["VAL_SERIES_START"] >= _d["TRAIN_START"], \
    f"Validation encoder would start on {VAL_SERIES_START}, before TRAIN_START."
_val_days = (_d["VAL_END"] - _d["VAL_START"]).days + 1
assert _val_days >= OUTPUT_CHUNK_LENGTH, \
    f"Scored validation period is {_val_days} days; one validation window scores {OUTPUT_CHUNK_LENGTH}."
print(f"ICL / OCL                : {INPUT_CHUNK_LENGTH} / {OUTPUT_CHUNK_LENGTH}")
print(f"Validation scored dates  : {VAL_START} -> {VAL_END} ({_val_days} days)")
print(f"Validation encoder starts: {VAL_SERIES_START}")
print(f"Validation windows       : {_val_days - OUTPUT_CHUNK_LENGTH + 1} per series")
print(f"Overlap with training targets: {'NONE' if _d['VAL_START'] > _d['TRAIN_END'] else 'YES -- val_loss partly measures memorisation'}")
print("NOTE: the scored validation dates end before Navratri/Diwali; Step 2 prints how many festive-weighted days they contain.")

## Section 1 — Snowflake session and source query
Only series listed in `VALID_SERIES_FOR_DAILY_MODELLING` are read. Nothing is downloaded yet;
`df` is a lazy Snowpark query.

In [ ]:
session = Session.builder.configs(Snowflake_configuration.ds1_role_json).create()
session.use_database("MOP_DATABASE")
session.use_schema("SOQ")

df = session.sql(f"""
    SELECT * FROM {TABLE_NAME}
    WHERE {group_col} IN (SELECT {group_col} FROM {VALID_SERIES_TBL})
""")

# strip quote characters Snowflake keeps in some column names
for old in df.columns:
    new = old.replace('"', "")
    if new != old:
        df = df.rename(old, new)

df = df.drop(*[c for c in DROP_COLS if c in df.columns])
df = df.with_column(group_col, F.replace(F.col(group_col), F.lit("<>"), F.lit("_")))
print(f"{len(df.columns)} columns after dropping {DROP_COLS}")

## Section 2 — Column roles
- **Static covariates**: every string column except the identifiers. One value per series.
- **Per-series covariates** (`SERIES_COV_COLS`): the two spend columns. They differ between series on the same date.
- **Shared continuous covariate** (`SHARED_CONT_COLS`): the petrol index. National, so identical for every series.
- **Binary covariates**: every remaining non-identifier column (festival blocks, single-day festivals, `IS_TUESDAY`, `IS_WEEKEND`, harvest windows, `POST_GST_CUT`). All are 0/1 and identical for every series.

`MODEL_NAME` should already be absent (dropped upstream because it is 1:1 with `MODEL_FAMILY`);
the assertions make that explicit so a schema change cannot quietly bring it back.

In [ ]:
id_cols = [time_col, group_col, target_col]

static_covariates = [
    f.name.replace('"', "") for f in df.schema
    if isinstance(f.datatype, StringType)
    and f.name.replace('"', "") not in NON_STATIC_STRING_COLS
]
all_cols = [c.replace('"', "") for c in df.columns]      # Snowpark quotes names like "N-16"

expected_cols = SERIES_COV_COLS + SHARED_CONT_COLS + NEW_BINARY_COLS
absent = [c for c in expected_cols if c not in all_cols]
assert not absent, f"These expected columns are not in {TABLE_NAME}: {absent}"

binary_covariates = [
    c for c in all_cols
    if c not in static_covariates and c not in id_cols
    and c not in CALENDAR_RAW_COLS and c not in NON_STATIC_STRING_COLS
    and c not in SERIES_COV_COLS and c not in SHARED_CONT_COLS
]
shared_check_cols = binary_covariates + SHARED_CONT_COLS      # must be identical across series on a date

assert "MODEL_FAMILY" in static_covariates, "MODEL_FAMILY must be a static covariate."
assert "MODEL_NAME" not in static_covariates, "MODEL_NAME is 1:1 with MODEL_FAMILY and should have been dropped."
assert dealer_col in static_covariates

print("static covariates   :", static_covariates)
print(f"binary covariates   : {len(binary_covariates)}")
print("shared continuous   :", SHARED_CONT_COLS)
print("per-series          :", SERIES_COV_COLS)

## Section 3 — Data-quality checks (run inside Snowflake)
Each check prints a one-line verdict. A failed check raises with an explanation; fix upstream rather than
patching locally.

| Check | Why it matters |
|---|---|
| 3a Target nulls / negatives / non-integers | The Negative-Binomial likelihood is undefined below 0 and for non-counts |
| 3b One row per series per day, no gaps (sales history) | A gap shifts every later day; the model indexes days by position |
| **3b2** Every series has rows from `TRAIN_START` to `FORECAST_END` | The per-series spend covariates are needed for every day, including the forecast window |
| 3c Static columns constant within a series | A series can have only one dealer/family/colour |
| 3d Static columns not null | A null would become its own category |
| 3e Binary + petrol columns identical across series on each date | Required for one shared calendar (Section 7) |
| **3f** Binary columns contain only 0/1; no nulls in any covariate; spend is never negative | A stray value such as 2 or -1 would silently act as a magnitude |

In [ ]:
hist = df.filter((F.col(time_col) >= TRAIN_START) & (F.col(time_col) <= VAL_END))
full = df.filter((F.col(time_col) >= TRAIN_START) & (F.col(time_col) <= FORECAST_END))

# --- 3a target ---------------------------------------------------------------
r = hist.select(
    F.count(F.lit(1)).alias("N_ROWS"),
    F.sum(F.when(F.col(target_col).is_null(), 1).otherwise(0)).alias("N_NULL"),
    F.sum(F.when(F.col(target_col) < 0, 1).otherwise(0)).alias("N_NEG"),
    F.sum(F.when(F.col(target_col) != F.round(F.col(target_col)), 1).otherwise(0)).alias("N_NON_INTEGER"),
).collect()[0].as_dict()
print("3a target:", r)
assert r["N_NULL"] == 0 and r["N_NEG"] == 0, "Target has nulls or negatives -- fix the source table."
assert r["N_NON_INTEGER"] == 0, "Target has non-integer values; a count likelihood needs integers."

# --- 3b one row per day, no gaps (sales history) ------------------------------
span = (hist.group_by(group_col)
            .agg(F.count(F.lit(1)).alias("N"), F.min(time_col).alias("D0"), F.max(time_col).alias("D1"))
            .with_column("EXPECTED", F.datediff("day", F.col("D0"), F.col("D1")) + 1))
r = span.select(
    F.count(F.lit(1)).alias("N_SERIES"),
    F.sum(F.when(F.col("N") < F.col("EXPECTED"), 1).otherwise(0)).alias("N_WITH_GAPS"),
    F.sum(F.when(F.col("N") > F.col("EXPECTED"), 1).otherwise(0)).alias("N_WITH_DUPLICATE_DAYS"),
    F.sum(F.when(F.col("D1") < F.lit(VAL_END), 1).otherwise(0)).alias("N_ENDING_BEFORE_VAL_END"),
).collect()[0].as_dict()
print("3b continuity:", r)
assert r["N_WITH_GAPS"] == 0 and r["N_WITH_DUPLICATE_DAYS"] == 0, "Series have missing or duplicate days."
if r["N_ENDING_BEFORE_VAL_END"]:
    print(f"   NOTE: {r['N_ENDING_BEFORE_VAL_END']} series stop before {VAL_END}; they cannot be forecast "
          f"from {FORECAST_START} and will be rejected in Step 2.")

# --- 3b2 every series has rows over the FULL span (covariates need the forecast window too) ---
span_full = (full.group_by(group_col)
                 .agg(F.count(F.lit(1)).alias("N"), F.min(time_col).alias("D0"), F.max(time_col).alias("D1"))
                 .with_column("EXPECTED", F.datediff("day", F.col("D0"), F.col("D1")) + 1))
r = span_full.select(
    F.count(F.lit(1)).alias("N_SERIES"),
    F.sum(F.when(F.col("N") != F.col("EXPECTED"), 1).otherwise(0)).alias("N_WITH_GAPS_OR_DUPLICATES"),
    F.sum(F.when(F.col("D0") != F.lit(TRAIN_START), 1).otherwise(0)).alias("N_NOT_STARTING_AT_TRAIN_START"),
    F.sum(F.when(F.col("D1") != F.lit(FORECAST_END), 1).otherwise(0)).alias("N_NOT_ENDING_AT_FORECAST_END"),
).collect()[0].as_dict()
print("3b2 full-span continuity:", r)
assert r["N_WITH_GAPS_OR_DUPLICATES"] == 0, "Some series have gaps or duplicates between TRAIN_START and FORECAST_END."
assert r["N_NOT_STARTING_AT_TRAIN_START"] == 0, f"Some series do not start on {TRAIN_START}."
assert r["N_NOT_ENDING_AT_FORECAST_END"] == 0, (
    f"Some series stop before {FORECAST_END}. The table needs a row (with the planned spend) for every series "
    f"on every forecast day.")

# --- 3c / 3d static columns --------------------------------------------------
per_series = hist.group_by(group_col).agg(*[F.count_distinct(F.col(c)).alias(f"S{i}")
                                            for i, c in enumerate(static_covariates)])
mx = per_series.select(*[F.max(F.col(f"S{i}")).alias(f"S{i}")
                         for i in range(len(static_covariates))]).collect()[0].as_dict()
not_constant = [c for i, c in enumerate(static_covariates) if (mx[f"S{i}"] or 0) > 1]
nulls = hist.select(*[F.sum(F.when(F.col(c).is_null(), 1).otherwise(0)).alias(f"S{i}")
                      for i, c in enumerate(static_covariates)]).collect()[0].as_dict()
has_null = [c for i, c in enumerate(static_covariates) if (nulls[f"S{i}"] or 0) > 0]
print("3c static columns varying within a series:", not_constant or "none")
print("3d static columns with nulls             :", has_null or "none")
assert not not_constant, f"These static columns change within a series: {not_constant}"
assert not has_null, f"These static columns contain nulls: {has_null}"

# --- 3e shared columns national ----------------------------------------------
cal_src = full                      # TRAIN_START -> FORECAST_END
per_date = cal_src.group_by(time_col).agg(*[F.count_distinct(F.col(c)).alias(f"C{i}")
                                            for i, c in enumerate(shared_check_cols)])
mx = per_date.select(*[F.max(F.col(f"C{i}")).alias(f"C{i}")
                       for i in range(len(shared_check_cols))]).collect()[0].as_dict()
regional = [c for i, c in enumerate(shared_check_cols) if (mx[f"C{i}"] or 0) > 1]
print("3e shared columns that differ across series on the same date:", regional or "none")
assert not regional, (
    f"{regional} vary across series on the same date (regional festivals? a regional petrol price?). "
    "A single shared calendar would apply one region's values to everyone. "
    "Move these to per-series covariates before continuing.")

# --- 3f value checks: binary columns, nulls, spend range ---------------------
aggs = []
for i, c in enumerate(binary_covariates):
    # counts every row whose value is not exactly 0 or 1 (nulls count too)
    aggs.append(F.sum(F.when(F.col(c).isin(0, 1), 0).otherwise(1)).alias(f"B{i}"))
cont_cols = SERIES_COV_COLS + SHARED_CONT_COLS
for i, c in enumerate(cont_cols):
    aggs.append(F.sum(F.when(F.col(c).is_null(), 1).otherwise(0)).alias(f"N{i}"))
    aggs.append(F.min(F.col(c)).alias(f"L{i}"))
    aggs.append(F.max(F.col(c)).alias(f"H{i}"))
r = full.select(*aggs).collect()[0].as_dict()

not_binary = {c: r[f"B{i}"] for i, c in enumerate(binary_covariates) if (r[f"B{i}"] or 0) > 0}
print("3f binary columns with values other than 0/1 (or null):", not_binary or "none")
assert not not_binary, (
    f"These columns were expected to be 0/1 but are not: {not_binary}. "
    "Fix upstream; do not clip locally.")
for i, c in enumerate(cont_cols):
    print(f"3f {c:30s} nulls={r[f'N{i}'] or 0:>3}  min={float(r[f'L{i}']):>10.2f}  max={float(r[f'H{i}']):>10.2f}")
    assert (r[f"N{i}"] or 0) == 0, (
        f"{c} has nulls between {TRAIN_START} and {FORECAST_END}. "
        "For the spend columns this usually means the Oct-Dec 2026 plan was not joined for some series.")
for i, c in enumerate(cont_cols):
    if c in SERIES_COV_COLS:
        assert float(r[f"L{i}"]) >= 0, f"{c} has negative values; log1p is undefined below -1 and spend cannot be negative."

## Section 4 — Category vocabularies (the code book for embeddings)
For each static column: the sorted list of distinct values, with `"__UNKNOWN__"` at position 0.
A category's **integer code is its position in this list**. Example: if
`vocab["MODEL_FAMILY"] = ["__UNKNOWN__", "GLAMOUR", "XOOM", ...]` then GLAMOUR → 1, XOOM → 2.

Why reserve 0: at prediction time a category never seen in training (a new dealer) maps to 0,
which is a valid row of the embedding table. The old `unknown_value=-1` is not a valid row and would crash.

The code is only an index into a table of learned vectors. Code 2 is not "twice" code 1. Step 2 has a test
that proves the model treats it that way.

In [ ]:
vocab = {}
for c in static_covariates:
    values = sorted(str(r[0]) for r in hist.select(F.col(c)).distinct().collect())
    vocab[c] = ["__UNKNOWN__"] + values
    print(f"  {c:20s} {len(values):5d} categories   e.g. {values[:3]}")

with open(vocab_path, "w") as f:
    json.dump(vocab, f, indent=1)
print(f"\nVocabulary written -> {vocab_path}")

## Section 5 — Scaling the three continuous features (decision and reasons)
**The pipeline has no scaler.** Every covariate in iteration 3 was already between -1 and 1 (0/1 flags, sin/cos,
`DAYS_TO_MONTH_END / 31`), and the target stays as raw counts under the Negative Binomial loss. Left raw, the three
new columns would enter the network 100 to 4,000 times larger than every other input. That is a problem for three reasons:
- the first layers would be dominated by those inputs, and the variable-selection softmax can lock onto the largest-magnitude input (the same failure that `RAKSHA_BANDHAN_DAYS` caused earlier);
- training uses `bf16-mixed` precision, which keeps only about 3 significant digits, so small month-to-month changes in a large number can be rounded away;
- gradient clipping is 0.1, so a few inputs with huge magnitudes make every step a clipped step.

| Feature | Raw summary (all rows, from your describe table) | Decision | Reason |
|---|---|---|---|
| `TOTAL_SPEND_PER_10K` | min 15.7 · median 99.7 · mean 310 · max 3,406 | `log1p`, then z-score | Right-skewed: the mean is 3.1× the median and the max is 34× the median. Without the log, a handful of very high values would squash everyone else toward zero. |
| `TOTAL_ADSTOCK_SPEND_PER_10K` | min 39.9 · median 345 · mean 603 · max 3,928 | `log1p`, then z-score | Milder skew (1.7× and 11×). Same treatment, so both spend features sit on one scale. |
| `PETROL_PRICE_INDEX` | 88.1 – 124.8 · median 96.1 · mean 99.6 · sd 11.7 | z-score only | A narrow band (sd is about 12% of the mean), so a log would change almost nothing. Only the offset of about 100 and the spread need removing. |
| 0/1 flags, sin/cos, days-to-month-end | already within -1 … 1 | none | |
| `NET_SALES` (target) | counts | none | Scored as counts by the Negative Binomial loss. |

**Constants come from training rows only** (`TRAIN_START` → `TRAIN_END`), so nothing about the validation or forecast
period leaks into the scaling. Validation and forecast values are transformed with the same constants. Planned spend
for Oct–Dec 2026 can therefore sit outside the range the model trained on; the cell below prints how far.

Spend is scaled **inside Snowflake** (the mean and standard deviation are computed there over all training rows), and the chunk
files already contain the scaled values. Petrol is scaled in Section 7 when the calendar is built.

In [ ]:
SCALING = {}

train_rows = df.filter((F.col(time_col) >= TRAIN_START) & (F.col(time_col) <= TRAIN_END))
log_expr = [F.ln(F.lit(1.0) + F.col(c)) for c in SERIES_COV_COLS]
r = train_rows.select(
    *[F.avg(e).alias(f"M{i}") for i, e in enumerate(log_expr)],
    *[F.stddev(e).alias(f"S{i}") for i, e in enumerate(log_expr)],
).collect()[0].as_dict()
for i, c in enumerate(SERIES_COV_COLS):
    mu, sd = float(r[f"M{i}"]), float(r[f"S{i}"])
    assert np.isfinite(mu) and np.isfinite(sd) and sd > 0, f"{c}: could not compute scaling constants ({mu}, {sd})."
    SCALING[c] = dict(transform="log1p_zscore", mean=mu, std=sd)
    print(f"{c:30s} log1p mean = {mu:.4f}   log1p std = {sd:.4f}")

# the lazy query every later download uses: spend columns replaced by their scaled values
df_model = df
for c in SERIES_COV_COLS:
    s = SCALING[c]
    df_model = df_model.with_column(
        c, (F.ln(F.lit(1.0) + F.col(c)) - F.lit(s["mean"])) / F.lit(s["std"]))

# --- verify: scaled training values should have mean ~0, std ~1; show how far val/forecast drift ---
windows = {"train": (TRAIN_START, TRAIN_END), "val": (VAL_START, VAL_END), "forecast": (FORECAST_START, FORECAST_END)}
summary = []
for name, (d0, d1) in windows.items():
    sub = df_model.filter((F.col(time_col) >= d0) & (F.col(time_col) <= d1))
    q = sub.select(
        *[F.avg(F.col(c)).alias(f"A{i}") for i, c in enumerate(SERIES_COV_COLS)],
        *[F.stddev(F.col(c)).alias(f"D{i}") for i, c in enumerate(SERIES_COV_COLS)],
        *[F.min(F.col(c)).alias(f"L{i}") for i, c in enumerate(SERIES_COV_COLS)],
        *[F.max(F.col(c)).alias(f"H{i}") for i, c in enumerate(SERIES_COV_COLS)],
    ).collect()[0].as_dict()
    for i, c in enumerate(SERIES_COV_COLS):
        summary.append(dict(window=name, feature=c, mean=float(q[f"A{i}"]), std=float(q[f"D{i}"]),
                            min=float(q[f"L{i}"]), max=float(q[f"H{i}"])))
scaled_summary = pd.DataFrame(summary)
print(scaled_summary.round(3).to_string(index=False))

for c in SERIES_COV_COLS:
    tr = scaled_summary[(scaled_summary.window == "train") & (scaled_summary.feature == c)].iloc[0]
    fc = scaled_summary[(scaled_summary.window == "forecast") & (scaled_summary.feature == c)].iloc[0]
    assert abs(tr["mean"]) < 1e-3 and abs(tr["std"] - 1) < 1e-3, f"{c}: scaled training values are not mean 0 / std 1."
    if fc["max"] > tr["max"] + 0.5 or fc["min"] < tr["min"] - 0.5:
        print(f"*** WARNING {c}: forecast-window values reach [{fc['min']:.2f}, {fc['max']:.2f}] "
              f"but training only covered [{tr['min']:.2f}, {tr['max']:.2f}] (in standard deviations). "
              f"The model will be extrapolating on this feature.")

## Section 6 — Download history and forecast-window covariates in dealer chunks
Rows from `TRAIN_START` to `FORECAST_END`, ~100 dealers per parquet file. Only the columns Step 2 needs are pulled:
keys, target, static columns and the two **scaled** spend columns. The festive flags and petrol come from the shared
calendar instead.

`NET_SALES` is empty (or zero) after `VAL_END`; Step 2 only reads sales up to `VAL_END`. The forecast-window rows are
downloaded only because the planned spend for those days lives on them.

The folder is **deleted and re-downloaded on every run**. That's slower than reusing files, but old chunks
(from different dates, columns, or an earlier version of the Snowflake table) can never be mixed in.

In [ ]:
import shutil

shutil.rmtree(local_train_dir, ignore_errors=True)
os.makedirs(local_train_dir)

model_rows = df_model.filter((F.col(time_col) >= TRAIN_START) & (F.col(time_col) <= FORECAST_END))
chunk_cols = [time_col, group_col, target_col] + static_covariates + SERIES_COV_COLS
dealers = sorted(r[0] for r in model_rows.select(dealer_col).distinct().collect())
chunks = [dealers[i:i + CHUNK_SIZE] for i in range(0, len(dealers), CHUNK_SIZE)]
print(f"{len(dealers)} dealers -> {len(chunks)} chunks")

for i, chunk in enumerate(chunks):
    part = model_rows.select(*chunk_cols).filter(F.col(dealer_col).isin(chunk)).to_pandas()
    part[target_col] = part[target_col].astype(np.float32)
    for c in SERIES_COV_COLS:
        part[c] = part[c].astype(np.float32)
    part.to_parquet(os.path.join(local_train_dir, f"chunk_{i:04d}.parquet"), index=False)
    print(f"  chunk {i+1}/{len(chunks)}: {len(part):,} rows, {part[group_col].nunique():,} series")
    del part
    gc.collect()

## Section 7 — Shared calendar (binary flags + petrol + calendar features)
**Why one shared calendar:** every binary flag, the petrol index and every calendar feature depends only on the date,
not the series. Section 3e confirmed every series has the same value on a given date. So the model gets one
1,370-row table and every series points to the same object. That is ~31K× less RAM than a copy per series,
and training and prediction provably read the same file. (The spend columns do differ by series, so Step 2 handles them separately.)

The calendar must reach `FORECAST_END`, because future covariates are needed for every forecast day.

Calendar features (kept from iteration 3):
- `DOW_SIN/COS` — weekday on a circle, so Sunday and Monday are neighbours.
- `IS_MONTH_END`, `IS_MONTH_START`, `DAYS_TO_MONTH_END` — the month-end retail push is a step, not a smooth cycle.
- Month-of-year and day-of-month are deliberately **excluded**: together they pin an exact calendar date and
  let the model copy last year's peak date instead of tracking Diwali (diagnosed in iteration 1).
- `YEAR` is excluded: 2026 values at forecast time would be out of range for anything the model learned.

`IS_TUESDAY` and `IS_WEEKEND` now come from the table. They overlap with `DOW_SIN/COS`; both are kept so this iteration
changes one thing at a time.

**Petrol.** It is z-scored with the mean and standard deviation of the training dates. The cell also prints the raw values in
the forecast window next to the last history value. If they are all identical to the `VAL_END` value, the Oct–Dec numbers
are a carried-forward assumption, not a projection, and the model is being told "no change" for the whole festive season.

In [ ]:
cal = (cal_src.group_by(time_col)
              .agg(*[F.max(F.col(c)).alias(f"C{i}") for i, c in enumerate(shared_check_cols)])
              .to_pandas())
cal.columns = [time_col] + shared_check_cols          # restore real names (aliases avoid quoting issues)
cal[time_col] = pd.to_datetime(cal[time_col])
cal = cal.sort_values(time_col).reset_index(drop=True)
cal[shared_check_cols] = cal[shared_check_cols].astype("float64")      # Snowflake can return Decimal objects

full_range = pd.date_range(TRAIN_START, FORECAST_END, freq=FREQ)
missing = full_range.difference(cal[time_col])
assert len(missing) == 0, (
    f"Calendar is missing {len(missing)} dates (first: {list(missing[:3])}). If the gap is at the end, "
    f"extend {TABLE_NAME} to {FORECAST_END}.")
assert cal[time_col].is_unique and len(cal) == len(full_range), "Calendar must have exactly one row per day."
assert cal[shared_check_cols].notna().all().all(), "Shared columns contain nulls."
assert cal[binary_covariates].isin([0.0, 1.0]).all().all(), "A binary column contains a value other than 0 or 1."

# --- POST_GST_CUT must switch 0 -> 1 once and stay 1 --------------------------
gst = cal.set_index(time_col)["POST_GST_CUT"]
assert gst.is_monotonic_increasing, "POST_GST_CUT must switch from 0 to 1 once and never go back."
assert (gst == 1).any() and (gst == 0).any(), "POST_GST_CUT never changes inside the date range."
first_one = gst[gst == 1].index.min()
print(f"POST_GST_CUT first equals 1 on {first_one.date()}"
      + ("" if first_one == pd.Timestamp("2025-09-22") else "   *** expected 2025-09-22 (the date the rate cut took effect) -- check the definition ***"))
assert (gst.loc[FORECAST_START:FORECAST_END] == 1).all(), "POST_GST_CUT should be 1 for the whole forecast window."

# --- petrol: z-score with training-date statistics ------------------------------
p_raw = cal["PETROL_PRICE_INDEX"].astype("float64")
is_train_day = cal[time_col] <= pd.Timestamp(TRAIN_END)
p_mu, p_sd = float(p_raw[is_train_day].mean()), float(p_raw[is_train_day].std(ddof=0))
assert np.isfinite(p_mu) and p_sd > 0, "Petrol index is constant over the training dates; it cannot be scaled."
SCALING["PETROL_PRICE_INDEX"] = dict(transform="zscore", mean=p_mu, std=p_sd)
cal["PETROL_PRICE_INDEX"] = ((p_raw - p_mu) / p_sd).astype(np.float32)
print(f"PETROL_PRICE_INDEX  train mean = {p_mu:.3f}  train std = {p_sd:.3f}")

in_fc = (cal[time_col] >= pd.Timestamp(FORECAST_START)) & (cal[time_col] <= pd.Timestamp(FORECAST_END))
fc_raw = p_raw[in_fc]
last_hist = float(p_raw[cal[time_col] == pd.Timestamp(VAL_END)].iloc[0])
print(f"Petrol, raw: last history value ({VAL_END}) = {last_hist:.2f}; forecast window has "
      f"{fc_raw.nunique()} distinct value(s), range {fc_raw.min():.2f} – {fc_raw.max():.2f}")
if fc_raw.nunique() == 1 and np.isclose(fc_raw.iloc[0], last_hist):
    print("*** NOTE: the forecast-window petrol index equals the last history value on every day -- "
          "it is carried forward, not projected. ***")
fc_z = cal.loc[in_fc, "PETROL_PRICE_INDEX"]
tr_z = cal.loc[is_train_day, "PETROL_PRICE_INDEX"]
if fc_z.max() > tr_z.max() + 0.5 or fc_z.min() < tr_z.min() - 0.5:
    print(f"*** WARNING: forecast-window petrol (z) [{fc_z.min():.2f}, {fc_z.max():.2f}] is outside the "
          f"training range [{tr_z.min():.2f}, {tr_z.max():.2f}]. ***")

# --- hand-built calendar features -----------------------------------------------
dt = cal[time_col].dt
cal["DOW_SIN"] = np.sin(2 * np.pi * dt.dayofweek / 7).astype(np.float32)
cal["DOW_COS"] = np.cos(2 * np.pi * dt.dayofweek / 7).astype(np.float32)
cal["IS_MONTH_END"]      = (dt.day >= dt.days_in_month - 2).astype(np.float32)
cal["IS_MONTH_START"]    = (dt.day <= 3).astype(np.float32)
cal["DAYS_TO_MONTH_END"] = ((dt.days_in_month - dt.day) / 31.0).astype(np.float32)
calendar_covariates = ["DOW_SIN", "DOW_COS", "IS_MONTH_END", "IS_MONTH_START", "DAYS_TO_MONTH_END"]

cal[binary_covariates] = cal[binary_covariates].astype(np.float32)
shared_covariates = binary_covariates + SHARED_CONT_COLS + calendar_covariates
future_covariates = shared_covariates + SERIES_COV_COLS      # the order the network will see (spend columns last)
cal[[time_col] + shared_covariates].to_parquet(calendar_path, index=False)

fc = cal[(cal[time_col] >= FORECAST_START) & (cal[time_col] <= FORECAST_END)]
print(f"\nCalendar: {cal[time_col].min().date()} -> {cal[time_col].max().date()} ({len(cal)} days)")
print(f"future covariates the model will see: {len(future_covariates)} "
      f"= {len(binary_covariates)} binary + {len(SHARED_CONT_COLS)} petrol + {len(calendar_covariates)} calendar + {len(SERIES_COV_COLS)} spend")
for anchor in ["N", "D"]:
    if anchor in fc.columns:
        print(f"  '{anchor}' falls on: {[d.date().isoformat() for d in fc.loc[fc[anchor] != 0, time_col]]}")

## Section 8 — Column roles for Steps 2 and 3

In [ ]:
roles = dict(
    time_col=time_col, group_col=group_col, dealer_col=dealer_col, target_col=target_col, freq=FREQ,
    static_covariates=static_covariates,
    binary_covariates=binary_covariates,
    shared_continuous_covariates=SHARED_CONT_COLS,
    calendar_covariates=calendar_covariates,
    shared_covariates=shared_covariates,
    series_covariates=SERIES_COV_COLS,
    future_covariates=future_covariates,
    scaling=SCALING,
    train_start=TRAIN_START, train_end=TRAIN_END,
    val_start=VAL_START, val_end=VAL_END, val_series_start=VAL_SERIES_START,
    forecast_start=FORECAST_START, forecast_end=FORECAST_END,
    input_chunk_length=INPUT_CHUNK_LENGTH, output_chunk_length=OUTPUT_CHUNK_LENGTH,
    vocab_path=vocab_path, calendar_path=calendar_path,
)
with open(roles_path, "w") as f:
    json.dump(roles, f, indent=2)
print(f"Roles written -> {roles_path}")
print("scaling constants:", json.dumps(SCALING, indent=1))

session.close()
print("Snowflake session closed.")